In [ ]:
# Adding necessary packages
from pathlib import Path
import re
import json
import numpy as np
import pandas as pd

# Using a try-except statement to create the file reader function
def read_file(file_path: Path) -> tuple[str,str]:
    """Read the files and return the text content and the encoding used"""
    try:
        file_text_content = file_path.read_text(encoding = "utf-8")
        encoding_used = "utf-8"
    except UnicodeDecodeError:
        file_text_content = file_path.read_text(encoding = "ISO-8859-1")
        encoding_used = "ISO-8859-1"
    
    return file_text_content, encoding_used


# Creating the log parser function
def parse_log(file_text_content: str) -> list[dict]:
    """ Parse the log text and return the valid transaction records"""
    
    # Creating an empty list for the records
    records = []
    
    # Storing the regex pattern
    pattern = re.compile(r"\[(.*?)\] \[(.*?)\] item=(.*?) price=(.*?) (.*?) \((.*?)\)")
    
    # Going through each line and storing matching expressions in a match variable
    for line in file_text_content.splitlines(): # Splitlines to go through each line
        match = pattern.fullmatch(line)
        
        if not match:
            continue
        
        # Collecting the key fields
        timestamp = match.group(1)
        register_id = match.group(2)
        item = match.group(3)
        price = match.group(4)
        currency = match.group(5)
        status = match.group(6)
        
        # Saving the key fields data in a dictionary
        record = {
            "timestamp": timestamp,
            "register_id": register_id,
            "item": item,
            "price": price,
            "currency": currency,
            "status": status,
        }
        
        # Appending the data to the records list
        records.append(record)
        
    return records


# Creating the log file finder function
def find_log_files(directory: str | Path) -> list[Path]:
    """Find all log files in a directory and in all subdirectories"""

    # Converting the directory to a Path object
    directory = Path(directory)
    
    # Creating an empty list to store the log files
    log_files = []

    # Iterating through the directory and its subdirectories
    for item in directory.iterdir():

        # Checking if the item is a directory, and using recursion if it is, adds the files to the list
        if item.is_dir():
            log_files.extend(find_log_files(item))

        # Checking if item is a file and if it has .log extension, adds it to the list
        elif item.is_file() and item.suffix == ".log":
            log_files.append(item)

    return log_files


# Creating the function to find active registers
def find_active_registers(data: object) -> set[str]:
    """Find all register IDs where active is true"""

    # Creating an empty set to store the active register IDs
    active_registers = set()

    # Checing is the data is a dictionary
    if isinstance(data, dict):

        # Checking if the register is active
        if data.get("active") is True:

            # Get the register ID
            register_id = data.get("id")

            # Checking if the register ID is a string
            if isinstance(register_id, str):
                active_registers.add(register_id)

        # Using recursion to go through the values of the dictionary and find active registers 
        for value in data.values():
            active_registers.update(find_active_registers(value))
    
    # Checking if the data is a list
    elif isinstance(data, list):
        
        # Using recursion to go through the items of the list and find active registers
        for item in data:
            active_registers.update(find_active_registers(item))

    return active_registers


# Create a function to remove price outliers and replace them with the median price
def remove_price_outliers(prices) -> np.ndarray:
    """Replace price outliers with the median price."""

    # Convert the input to a 1D numpy float array
    prices = np.asarray(prices, dtype=float).flatten()

    # Calculate the mean and standard deviation
    mean = np.mean(prices)
    std = np.std(prices)

    # Identify outliers (values more than 3 standard deviations from the mean)
    outliers = np.abs(prices - mean) > 3 * std

    # Calculate the median of valid prices
    valid_prices = prices[~outliers]
    median = np.median(valid_prices)

    # Replace outliers with the median price
    prices[outliers] = median

    return prices


# Create a function to compute the refund ratio
def compute_refund_ratio(
        sales_count: np.ndarray,
        refund_count: np.ndarray
) -> np.ndarray:
    """Computes the refund ratio for each element."""

    # Calculate the refund ratio element wise, handling division by zero
    refund_ratio = np.divide(
        refund_count,
        sales_count + 1e-6
    )  # Adding a small value to avoid division by zero

    return refund_ratio


# Converting to pandas dataframe
def transactions_to_dataframe(records: list[dict]) -> pd.DataFrame:
    """Convert transaction records to a chronologically sorted DataFrame"""
    
    # Converting the list of dictionaries into a df
    df = pd.DataFrame(records)
    
    # Converting the timestamp column to datetime
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    
    # Set timestamp as the index
    df = df.set_index("timestamp")
    
    # Sort the dataframe chronologically by timestamp
    df = df.sort_index()
    
    return df

# Converting price column to numeric values and calculating the sums
def hourly_revenue_by_item(df: pd.DataFrame) -> pd.DataFrame:
    """Group transactions by item and calculate total price per hour"""
    
    # Convert price to numeric values
    df["price"] = pd.to_numeric(df["price"])
    
    # Group by item and resample into one-hour windows
    hourly_revenue = df.groupby("item").resample("1h")["price"].sum()
    
    return hourly_revenue


def revenue_pivot_table(df: pd.DataFrame) -> pd.DataFrame:
    """Create a pivot table of total revenue by item and transaction status."""

    # Convert price to numeric values
    df["price"] = pd.to_numeric(df["price"])

    # Create the pivot table
    pivot_table = pd.pivot_table(
        df,
        values="price",
        index="item",
        columns="status",
        aggfunc="sum",
        fill_value=0
    )

    return pivot_table



def refund_ratio_by_item(df: pd.DataFrame) -> pd.Series:
    """Calculate the refund ratio for each item."""

    # Count sales and refunds for each item
    sales_count = df[df["status"] == "SUCCESS"].groupby("item").size()
    refund_count = df[df["status"] == "REFUND"].groupby("item").size()

    # Align the counts so all items are included
    sales_count = sales_count.add(refund_count * 0, fill_value=0)
    refund_count = refund_count.reindex(sales_count.index, fill_value=0)

    # Calculate the refund ratio using the function from task 3
    refund_ratio = compute_refund_ratio(
        sales_count.to_numpy(),
        refund_count.to_numpy()
    )

    return pd.Series(refund_ratio, index=sales_count.index, name="refund_ratio")





In [ ]:
def run_workflow(data_dir: Path) -> None:
    """Run the transaction-processing workflow and save all reports"""

    # 1. Find log files and identify active registers
    log_files = find_log_files(data_dir)

    with (data_dir / "store_config.json").open(encoding = "utf-8") as file:
        config_data = json.load(file)

    active_registers = find_active_registers(config_data)

    # 2. Read and parse all log files
    all_records = []

    for log_file in log_files:
        file_text_content, encoding_used = read_file(log_file)
        records = parse_log(file_text_content)

        # Read and parse all log files
        all_records = []

        for log_file in log_files:
            file_text_content, encoding_used = read_file(log_file)
            records = parse_log(file_text_content)
            all_records.extend(records)

    # 3. Create a Dataframe containing all parsed transactions
    df = transactions_to_dataframe(all_records)

    # 4. Clean price outliers
    df["price"] = remove_price_outliers((df["price"]).to_numpy())

    # 5. Correct prices for NOK transactions
    nok_mask = df["currency"] == "NOK"
    df.loc[nok_mask, "price"] *= 10.0

    # Round prices to two decimal places
    df["price"] = df["price"].round(2)

    # 6. Generate resampled reports, pivot tables and refund ratios
    hourly_revenue = hourly_revenue_by_item(df)
    status_pivot_table = revenue_pivot_table(df)
    refund_ratio = refund_ratio_by_item(df)

    # 7. Save all required output files
    output_dir = data_dir / "output"
    output_dir.mkdir(parents=True, exist_ok=True)

    # Save cleaned and corrected transactions to CSV
    df.to_csv(output_dir / "cleaned_transactions.csv", index=True)

    # Save hourly revenue report to JSON
    hourly_revenue.to_json(
        output_dir / "hourly_revenue.json", 
        orient="index",
        indent=2,
        )

    # Revenue grouped by item and transaction status
    status_pivot_table.to_csv(output_dir / "status_pivot_table.csv")

    # Refund ratio for each item
    refund_ratio.to_csv(
        output_dir / "item_refund_ratios.csv",
        float_format="%.2f"
    )


if __name__ == "__main__":
    data_dir = Path("data")
    run_workflow(data_dir)